# 🏥 TrialMatch AI — Medical NER Fine-Tuning

**Task**: Token classification (BIO tagging) — extract medical entities from text

**Entities**: AGE, SEX, CONDITION, STAGE, BIOMARKER, MEDICATION, ECOG, LAB_VALUE, PRIOR_THERAPY, RESPONSE, COMORBIDITY, ALLERGY

**Method**: LoRA (PEFT) on BioBERT

**Runtime**: T4 GPU required

In [ ]:
# Step 1: Install
!pip install -q transformers datasets evaluate accelerate peft scikit-learn seqeval

In [ ]:
# Step 2: GPU check
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Upload train.jsonl
from google.colab import files
print("Upload train.jsonl from fine_tuning/data/medical_ner/")
uploaded = files.upload()

In [ ]:
# Step 4: Config
ENTITY_LABELS = [
    "O",
    "B-AGE", "I-AGE", "B-SEX", "I-SEX",
    "B-CONDITION", "I-CONDITION", "B-STAGE", "I-STAGE",
    "B-BIOMARKER", "I-BIOMARKER", "B-MEDICATION", "I-MEDICATION",
    "B-LAB_VALUE", "I-LAB_VALUE", "B-ECOG", "I-ECOG",
    "B-PRIOR_THERAPY", "I-PRIOR_THERAPY", "B-RESPONSE", "I-RESPONSE",
    "B-COMORBIDITY", "I-COMORBIDITY", "B-ALLERGY", "I-ALLERGY",
    "B-METASTATIC_SITE", "I-METASTATIC_SITE",
    "B-DRUG_INTERACTION", "I-DRUG_INTERACTION",
]
label2id = {l: i for i, l in enumerate(ENTITY_LABELS)}
id2label = {i: l for i, l in enumerate(ENTITY_LABELS)}

CONFIG = {
    "base_model": "dmis-lab/biobert-v1.1",
    "max_seq_length": 256,
    "num_labels": len(ENTITY_LABELS),
    "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
    "target_modules": ["query", "value"],
    "num_epochs": 20, "batch_size": 16, "learning_rate": 2e-4,
    "weight_decay": 0.01, "warmup_ratio": 0.1,
    "output_dir": "./medical_ner_lora",
    "merged_dir": "./medical_ner_merged",
}
print(f"✅ {len(ENTITY_LABELS)} BIO labels, {(len(ENTITY_LABELS)-1)//2} entity types")

In [ ]:
# Step 5: Load data and convert to BIO format
import json
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])

raw = []
with open("train.jsonl") as f:
    for line in f: raw.append(json.loads(line))
print(f"Loaded {len(raw)} examples")

def convert_to_bio(text, entities, tokenizer):
    encoding = tokenizer(text, padding="max_length", truncation=True,
                         max_length=CONFIG["max_seq_length"], return_offsets_mapping=True)
    offsets = encoding["offset_mapping"]
    labels = [-100] * len(offsets)
    sorted_ents = sorted(entities, key=lambda e: e["start"])
    
    for i, (start, end) in enumerate(offsets):
        if start == 0 and end == 0:
            labels[i] = -100
            continue
        labels[i] = label2id["O"]
        for ent in sorted_ents:
            if start >= ent["start"] and end <= ent["end"]:
                if start == ent["start"] or (i > 0 and offsets[i-1][1] <= ent["start"]):
                    bio = f"B-{ent['label']}"
                else:
                    bio = f"I-{ent['label']}"
                if bio in label2id:
                    labels[i] = label2id[bio]
                break
    
    encoding.pop("offset_mapping")
    encoding["labels"] = labels
    return encoding

processed = [convert_to_bio(ex["text"], ex["entities"], tokenizer) for ex in raw]
print(f"✅ Converted {len(processed)} examples to BIO format")

In [ ]:
# Step 6: Create datasets
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(processed, test_size=0.2, random_state=42)
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
train_dataset.set_format("torch")
val_dataset.set_format("torch")
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Step 7: Load model + LoRA
from transformers import AutoModelForTokenClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForTokenClassification.from_pretrained(
    CONFIG["base_model"], num_labels=CONFIG["num_labels"],
    id2label=id2label, label2id=label2id, torch_dtype=torch.float16,
).to("cuda")

lora_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS, r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"], bias="none",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = 100 * trainable / total
print(f"✅ LoRA applied — Trainable: {trainable:,} ({pct:.2f}%)")

In [ ]:
# Step 8: Metrics
import numpy as np
from seqeval.metrics import f1_score as seq_f1

def compute_ner_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_labels, pred_labels = [], []
    for pred_seq, label_seq in zip(preds, labels):
        t, p = [], []
        for pr, la in zip(pred_seq, label_seq):
            if la != -100:
                t.append(id2label.get(int(la), "O"))
                p.append(id2label.get(int(pr), "O"))
        true_labels.append(t)
        pred_labels.append(p)
    return {"f1": seq_f1(true_labels, pred_labels, average="weighted", zero_division=0)}

print("✅ Metrics ready")

In [ ]:
# Step 9: Train
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_ratio=CONFIG["warmup_ratio"],
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="f1",
    greater_is_better=True, logging_steps=10,
    fp16=False, optim="adamw_torch",
    report_to="none", remove_unused_columns=False,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    compute_metrics=compute_ner_metrics,
)

print("🚀 Training NER...")
train_result = trainer.train()
print(f"\n✅ Done! Loss: {train_result.training_loss:.4f}")

In [ ]:
# Step 10: Evaluate
eval_results = trainer.evaluate()
print("📊 NER RESULTS")
for k, v in eval_results.items():
    print(f"   {k}: {v:.4f}" if isinstance(v, float) else f"   {k}: {v}")

In [ ]:
# Step 11: Merge + Save
from peft import PeftModel
import os

base_model = AutoModelForTokenClassification.from_pretrained(
    CONFIG["base_model"], num_labels=CONFIG["num_labels"],
    id2label=id2label, label2id=label2id, torch_dtype=torch.float16,
)
checkpoints = [d for d in os.listdir(CONFIG["output_dir"]) if d.startswith("checkpoint-")]
best = os.path.join(CONFIG["output_dir"], sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]) if checkpoints else CONFIG["output_dir"]

merged = PeftModel.from_pretrained(base_model, best).merge_and_unload()
merged.save_pretrained(CONFIG["merged_dir"])
tokenizer.save_pretrained(CONFIG["merged_dir"])

import json
metrics = {"model": CONFIG["base_model"], "method": "LoRA (PEFT)", "task": "NER",
           "entity_types": 14, "lora_rank": CONFIG["lora_r"],
           "trainable_pct": round(pct, 2), "train_loss": round(train_result.training_loss, 4),
           "eval": {k: round(v, 4) if isinstance(v, float) else v for k, v in eval_results.items()},
           "train_samples": len(train_dataset)}
with open(f"{CONFIG['merged_dir']}/training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"✅ Saved to {CONFIG['merged_dir']}")
print(json.dumps(metrics, indent=2))

In [ ]:
# Step 12: Test inference
from transformers import pipeline as hf_pipeline

ner = hf_pipeline("ner", model=CONFIG["merged_dir"], tokenizer=CONFIG["merged_dir"],
                  aggregation_strategy="simple", device=0)

tests = [
    "62-year-old male with stage IIIA NSCLC, EGFR positive, on Osimertinib",
    "48yo female HER2+ breast cancer, ECOG 0, prior Trastuzumab 6 cycles",
    "Allergic to Penicillin, comorbidities include type 2 diabetes and CKD",
]
print("\n🧪 NER TEST")
for text in tests:
    results = ner(text)
    print(f"\n  {text}")
    for e in results:
        print(f"    [{e['entity_group']:15s}] {e['word']:20s} ({e['score']:.3f})")

In [ ]:
# Step 13: Download
import shutil
shutil.make_archive("medical_ner_v1", "zip", ".", CONFIG["merged_dir"])
print("📦 Files:")
for f in sorted(os.listdir(CONFIG["merged_dir"])):
    print(f"   {f} ({os.path.getsize(os.path.join(CONFIG['merged_dir'], f))/1024:.1f} KB)")
print("\n🎯 Unzip into: trialmatch-ai/fine_tuning/models/medical_ner/")
files.download("medical_ner_v1.zip")